# Avito: лексические бейзлайны

Все методы оцениваются на валидации с новыми формулировками. Получение top-k
не зависит от порядка строк корпуса: при одинаковой оценке используется
`item_id`.


In [ ]:
%pip install -q pymorphy3==2.0.6


In [ ]:
from functools import lru_cache
import gc
import re
import time
import unicodedata

import numpy as np
import pandas as pd
import pymorphy3
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit

SEED = 42


In [ ]:
from pathlib import Path
import os

# Для локального запуска достаточно изменить PROJECT_DIR или задать
# AVITO_PROJECT_DIR / AVITO_DATA_DIR / AVITO_OUTPUT_DIR.
KAGGLE_DATA_DIR = Path("/kaggle/input/datasets/yuriymagus/avito-nlp-interns")
DEFAULT_PROJECT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

PROJECT_DIR = Path(os.environ.get("AVITO_PROJECT_DIR", DEFAULT_PROJECT_DIR)).expanduser().resolve()
DATA_DIR = Path(
    os.environ.get(
        "AVITO_DATA_DIR",
        KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else PROJECT_DIR / "data",
    )
).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("AVITO_OUTPUT_DIR", PROJECT_DIR)).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = (
    "train.parquet",
    "benchmark_queries.parquet",
    "benchmark_items.parquet",
)
missing_files = [name for name in REQUIRED_FILES if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Не найдены файлы {missing_files} в {DATA_DIR}. "
        "Измените DATA_DIR в первой ячейке."
    )

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"DATA_DIR:    {DATA_DIR}")
print(f"OUTPUT_DIR:  {OUTPUT_DIR}")


In [ ]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

print(f"train:   {train.shape}")
print(f"queries: {queries.shape}")
print(f"items:   {items.shape}")


In [ ]:
def normalize_query_group(values: pd.Series) -> pd.Series:
    return (
        values.astype("string")
        .str.casefold()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


train_query_groups = normalize_query_group(train["search_query"])

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED,
)
fit_idx, validation_pool_idx = next(
    splitter.split(train, groups=train_query_groups)
)

fit_events = train.iloc[fit_idx].copy()
validation_pool = train.iloc[validation_pool_idx].copy()
fit_groups = set(train_query_groups.iloc[fit_idx])

validation_unique = (
    validation_pool
    .assign(
        query_group=train_query_groups.iloc[validation_pool_idx].to_numpy()
    )
    .sample(frac=1, random_state=SEED)
    .drop_duplicates("query_group")
    .copy()
)

validation_unique["has_params"] = (
    validation_unique["search_infm_params_text"]
    .astype("string")
    .str.strip()
    .ne("")
)
benchmark_has_params = (
    queries["search_infm_params_text"]
    .astype("string")
    .str.strip()
    .ne("")
)

n_with_params = int(benchmark_has_params.sum())
n_without_params = len(queries) - n_with_params

validation_events = pd.concat(
    [
        validation_unique.loc[validation_unique["has_params"]]
        .sample(n=n_with_params, random_state=SEED),
        validation_unique.loc[~validation_unique["has_params"]]
        .sample(n=n_without_params, random_state=SEED),
    ],
    ignore_index=True,
).sample(frac=1, random_state=SEED).reset_index(drop=True)

assert len(validation_events) == len(queries)
assert validation_events["query_group"].is_unique
assert set(validation_events["query_group"]).isdisjoint(fit_groups)

print(f"fit events:        {len(fit_events):,}")
print(f"validation events: {len(validation_events):,}")


In [ ]:
ITEM_COLUMNS = [column for column in train.columns if column.startswith("item_")]
CORPUS_SIZE = len(items)

train_items = (
    train[ITEM_COLUMNS]
    .drop_duplicates("item_id")
    .copy()
)

validation_target_ids = set(
    validation_events["item_id"].astype("string")
)

required_items = train_items.loc[
    train_items["item_id"].astype("string").isin(validation_target_ids)
]
other_items = train_items.loc[
    ~train_items["item_id"].astype("string").isin(validation_target_ids)
]

n_other = CORPUS_SIZE - len(required_items)
assert n_other >= 0 and len(other_items) >= n_other

# Состав корпуса по-прежнему содержит все таргеты, но его порядок не кодирует таргет.
validation_items = (
    pd.concat(
        [
            required_items,
            other_items.sample(n=n_other, random_state=SEED),
        ],
        ignore_index=True,
    )
    .assign(__stable_item_id=lambda frame: frame["item_id"].astype("string"))
    .sort_values("__stable_item_id", kind="mergesort")
    .drop(columns="__stable_item_id")
    .reset_index(drop=True)
)

validation_corpus_ids = set(validation_items["item_id"].astype("string"))

assert len(validation_items) == CORPUS_SIZE
assert validation_items["item_id"].is_unique
assert validation_target_ids <= validation_corpus_ids
assert all(column.startswith("item_") for column in validation_items.columns)

print(f"валидационный корпус: {len(validation_items):,}")
print(f"обязательных таргетов: {len(required_items):,}")
print("Порядок корпуса: stable item_id; search_* columns отсутствуют")


In [ ]:
TOKEN_PATTERN = re.compile(r"[0-9a-zа-я]+")
CYRILLIC_TOKEN_PATTERN = re.compile(r"^[а-я]+$")
morph = pymorphy3.MorphAnalyzer()


def normalize_lexical_text(value: object) -> str:
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = text.casefold().replace("ё", "е")
    return " ".join(TOKEN_PATTERN.findall(text))


@lru_cache(maxsize=500_000)
def lemmatize_token(token: str) -> str:
    if not CYRILLIC_TOKEN_PATTERN.fullmatch(token):
        return token
    return morph.parse(token)[0].normal_form.replace("ё", "е")


def lemmatize_text(value: object) -> str:
    normalized = normalize_lexical_text(value)
    return " ".join(lemmatize_token(token) for token in normalized.split())


def normalize_identifier(values: pd.Series) -> np.ndarray:
    return (
        pd.to_numeric(values, errors="coerce")
        .astype("Int64")
        .astype("string")
        .fillna("<NA>")
        .to_numpy()
    )


def make_item_tie_keys(item_ids: np.ndarray) -> np.ndarray:
    '''Целочисленный ключ сохраняет лексикографический порядок item_id.'''
    item_ids = np.asarray(item_ids, dtype=str)
    order = np.argsort(item_ids, kind="mergesort")
    keys = np.empty(len(item_ids), dtype=np.int64)
    keys[order] = np.arange(len(item_ids), dtype=np.int64)
    return keys


In [ ]:
item_ids = validation_items["item_id"].astype("string").to_numpy()
target_ids = validation_events["item_id"].astype("string").to_numpy()
item_tie_keys = make_item_tie_keys(item_ids)


In [ ]:
def sparse_top_k(
    query_vectors,
    item_vectors,
    item_tie_keys: np.ndarray,
    k: int = 100,
    batch_size: int = 16,
) -> tuple[list[np.ndarray], np.ndarray]:
    """Точный top-k: score по убыванию, затем item_id по возрастанию."""
    rankings = []
    nonzero_counts = np.zeros(query_vectors.shape[0], dtype=np.int32)

    for batch_start in range(0, query_vectors.shape[0], batch_size):
        batch_end = min(batch_start + batch_size, query_vectors.shape[0])
        scores = (query_vectors[batch_start:batch_end] @ item_vectors.T).tocsr()

        for local_row in range(scores.shape[0]):
            row_start = scores.indptr[local_row]
            row_end = scores.indptr[local_row + 1]
            indices = scores.indices[row_start:row_end]
            values = scores.data[row_start:row_end]
            nonzero_counts[batch_start + local_row] = len(indices)

            if len(indices) == 0:
                rankings.append(np.empty(0, dtype=np.int32))
                continue

            order = np.lexsort((item_tie_keys[indices], -values))
            rankings.append(indices[order[: min(k, len(order))]].astype(np.int32))

    return rankings, nonzero_counts


def calculate_hits(rankings: list[np.ndarray], k: int) -> np.ndarray:
    return np.fromiter(
        (
            target_id in item_ids[ranking[:k]]
            for target_id, ranking in zip(target_ids, rankings)
        ),
        dtype=bool,
        count=len(target_ids),
    )


## 1. TF-IDF по заголовку


In [ ]:
item_titles = validation_items["item_title_raw"].fillna("").astype(str).tolist()
query_texts = validation_events["search_query"].fillna("").astype(str).tolist()

tfidf_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b\w+\b",
    lowercase=True,
    sublinear_tf=True,
    dtype=np.float32,
)
word_items = tfidf_word.fit_transform(item_titles)
word_queries = tfidf_word.transform(query_texts)
word_rankings, word_nonzero = sparse_top_k(
    word_queries, word_items, item_tie_keys, k=100
)

tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    sublinear_tf=True,
    dtype=np.float32,
)
char_items = tfidf_char.fit_transform(item_titles)
char_queries = tfidf_char.transform(query_texts)
char_rankings, char_nonzero = sparse_top_k(
    char_queries, char_items, item_tie_keys, k=100, batch_size=2
)

display(pd.DataFrame({
    "HitRate@10": [calculate_hits(word_rankings, 10).mean(), calculate_hits(char_rankings, 10).mean()],
    "HitRate@50": [calculate_hits(word_rankings, 50).mean(), calculate_hits(char_rankings, 50).mean()],
    "HitRate@100": [calculate_hits(word_rankings, 100).mean(), calculate_hits(char_rankings, 100).mean()],
}, index=["word TF-IDF", "char TF-IDF"]).round(4))


## 2. Лемматизация и BM25 по заголовку


In [ ]:
lemma_item_titles = [lemmatize_text(value) for value in validation_items["item_title_raw"]]
lemma_queries = [lemmatize_text(value) for value in validation_events["search_query"]]

lemma_tfidf = TfidfVectorizer(
    analyzer="word",
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    sublinear_tf=True,
    dtype=np.float32,
)
lemma_tfidf_items = lemma_tfidf.fit_transform(lemma_item_titles)
lemma_tfidf_queries = lemma_tfidf.transform(lemma_queries)
lemma_tfidf_rankings, _ = sparse_top_k(
    lemma_tfidf_queries, lemma_tfidf_items, item_tie_keys, k=100
)


In [ ]:
def build_bm25_matrix(count_matrix: csr_matrix, k1: float = 1.2, b: float = 0.75):
    counts = count_matrix.tocsr().astype(np.float32, copy=True)
    n_documents, n_terms = counts.shape
    document_frequency = np.bincount(counts.indices, minlength=n_terms).astype(np.float32)
    idf = np.log1p(
        (n_documents - document_frequency + 0.5)
        / (document_frequency + 0.5)
    ).astype(np.float32)
    lengths = np.asarray(counts.sum(axis=1)).ravel().astype(np.float32)
    average_length = float(lengths.mean())
    rows = np.repeat(np.arange(n_documents), np.diff(counts.indptr))
    normalization = k1 * (1.0 - b + b * lengths[rows] / average_length)
    counts.data = (
        idf[counts.indices] * counts.data * (k1 + 1.0)
        / (counts.data + normalization)
    )
    return counts


title_vectorizer = CountVectorizer(
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False,
    dtype=np.float32,
)
title_counts = title_vectorizer.fit_transform(lemma_item_titles)
title_query_counts = title_vectorizer.transform(lemma_queries).tocsr()
title_query_counts.data.fill(1.0)
title_bm25 = build_bm25_matrix(title_counts)
title_bm25_rankings, _ = sparse_top_k(
    title_query_counts, title_bm25, item_tie_keys, k=100
)

display(pd.DataFrame({
    "HitRate@10": [calculate_hits(lemma_tfidf_rankings, 10).mean(), calculate_hits(title_bm25_rankings, 10).mean()],
    "HitRate@50": [calculate_hits(lemma_tfidf_rankings, 50).mean(), calculate_hits(title_bm25_rankings, 50).mean()],
    "HitRate@100": [calculate_hits(lemma_tfidf_rankings, 100).mean(), calculate_hits(title_bm25_rankings, 100).mean()],
}, index=["lemma TF-IDF", "lemma BM25"]).round(4))


## 3. BM25F по заголовку и описанию


In [ ]:
def idf_from_document_frequency(
    document_frequency: np.ndarray,
    n_documents: int,
) -> np.ndarray:
    return np.log1p(
        (n_documents - document_frequency + 0.5)
        / (document_frequency + 0.5)
    ).astype(np.float32)


def normalize_field_term_frequency(
    count_matrix: csr_matrix,
    b: float = 0.75,
) -> csr_matrix:
    '''Нормализует поле на независимой копии и не меняет count_matrix.'''
    matrix = count_matrix.tocsr(copy=True).astype(np.float32, copy=False)
    field_lengths = np.asarray(matrix.sum(axis=1)).ravel().astype(np.float32)
    average_field_length = float(field_lengths.mean())

    if average_field_length == 0.0:
        return matrix

    row_normalization = 1.0 / (
        1.0 - b + b * field_lengths / average_field_length
    )
    matrix.data *= np.repeat(row_normalization, np.diff(matrix.indptr))
    return matrix


def build_bm25f_matrix(
    title_term_frequency: csr_matrix,
    description_term_frequency: csr_matrix,
    inverse_document_frequency: np.ndarray,
    title_weight: float = 1.0,
    description_weight: float = 1.0 / 3.0,
    k1: float = 1.2,
) -> csr_matrix:
    '''B8: веса title=1, description=1/3.'''
    combined_tf = (
        title_term_frequency.multiply(title_weight)
        + description_term_frequency.multiply(description_weight)
    ).tocsr()
    combined_tf.sum_duplicates()
    combined_tf.eliminate_zeros()

    combined_tf.data = (
        combined_tf.data * (k1 + 1.0)
        / (combined_tf.data + k1)
    )
    combined_tf.data *= inverse_document_frequency[combined_tf.indices]
    return combined_tf


In [ ]:
lemma_descriptions = [
    lemmatize_text(value) for value in validation_items["item_description_raw"]
]
field_vectorizer = CountVectorizer(
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False,
    dtype=np.float32,
)
combined_counts = field_vectorizer.fit_transform(
    f"{title} {description}"
    for title, description in zip(lemma_item_titles, lemma_descriptions)
).tocsr()
n_documents, n_terms = combined_counts.shape
document_frequency = np.bincount(
    combined_counts.indices, minlength=n_terms
).astype(np.float32)
field_idf = idf_from_document_frequency(document_frequency, n_documents)

title_counts = field_vectorizer.transform(lemma_item_titles).tocsr()
description_counts = field_vectorizer.transform(lemma_descriptions).tocsr()
field_queries = field_vectorizer.transform(lemma_queries).tocsr()
field_queries.data.fill(1.0)
title_tf = normalize_field_term_frequency(title_counts)
description_tf = normalize_field_term_frequency(description_counts)

weight_configs = {
    "title:description = 1:1": (1.0, 1.0),
    "title:description = 3:1": (1.0, 1.0 / 3.0),
    "title:description = 5:1": (1.0, 1.0 / 5.0),
}
bm25f_rankings = {}
bm25f_matrices = {}
rows = []
for name, (title_weight, description_weight) in weight_configs.items():
    matrix = build_bm25f_matrix(
        title_tf, description_tf, field_idf,
        title_weight=title_weight,
        description_weight=description_weight,
        k1=1.2,
    )
    rankings, _ = sparse_top_k(field_queries, matrix, item_tie_keys, k=100)
    bm25f_rankings[name] = rankings
    bm25f_matrices[name] = matrix
    rows.append({
        "model": name,
        "HitRate@10": calculate_hits(rankings, 10).mean(),
        "HitRate@50": calculate_hits(rankings, 50).mean(),
        "HitRate@100": calculate_hits(rankings, 100).mean(),
    })
display(pd.DataFrame(rows).set_index("model").round(4))


## 4. Глубина поиска и точная локация


In [ ]:
def select_exact_top_k(
    indices: np.ndarray,
    values: np.ndarray,
    item_tie_keys: np.ndarray,
    k: int,
    mask: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    '''Детерминированный top-k с учётом равных значений на границе.'''
    if mask is not None:
        indices = indices[mask]
        values = values[mask]

    if len(indices) == 0:
        return (
            np.empty(0, dtype=np.int32),
            np.empty(0, dtype=np.float32),
        )

    order = np.lexsort((item_tie_keys[indices], -values))
    selected = order[: min(k, len(order))]
    return (
        indices[selected].astype(np.int32, copy=False),
        values[selected].astype(np.float32, copy=False),
    )


def retrieve_global_and_location(
    query_vectors: csr_matrix,
    item_vectors: csr_matrix,
    query_locations: np.ndarray,
    item_locations: np.ndarray,
    item_tie_keys: np.ndarray,
    global_k: int = 200,
    location_k: int = 100,
    batch_size: int = 16,
) -> dict[str, list[np.ndarray] | np.ndarray]:
    '''За один расчёт строит глобальную выдачу и выдачу по локации.'''
    result = {
        "global_indices": [],
        "global_scores": [],
        "location_indices": [],
        "location_scores": [],
        "nonzero_counts": np.zeros(query_vectors.shape[0], dtype=np.int32),
    }

    for batch_start in range(0, query_vectors.shape[0], batch_size):
        batch_end = min(batch_start + batch_size, query_vectors.shape[0])
        scores = (query_vectors[batch_start:batch_end] @ item_vectors.T).tocsr()

        for local_row in range(scores.shape[0]):
            query_position = batch_start + local_row
            row_start = scores.indptr[local_row]
            row_end = scores.indptr[local_row + 1]
            indices = scores.indices[row_start:row_end]
            values = scores.data[row_start:row_end]
            result["nonzero_counts"][query_position] = len(indices)

            global_indices, global_scores = select_exact_top_k(
                indices, values, item_tie_keys, global_k
            )
            location_mask = (
                item_locations[indices] == query_locations[query_position]
            )
            location_indices, location_scores = select_exact_top_k(
                indices,
                values,
                item_tie_keys,
                location_k,
                mask=location_mask,
            )

            result["global_indices"].append(global_indices)
            result["global_scores"].append(global_scores)
            result["location_indices"].append(location_indices)
            result["location_scores"].append(location_scores)

    return result


In [ ]:
def build_fallback_orders(
    item_frame: pd.DataFrame,
    interaction_events: pd.DataFrame,
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    '''Резервный порядок: история → рейтинг → отзывы → item_id.'''
    item_ids = item_frame["item_id"].astype("string")
    history = (
        interaction_events["item_id"].astype("string").value_counts()
    )
    frame = pd.DataFrame({
        "item_index": np.arange(len(item_frame), dtype=np.int32),
        "item_id": item_ids.to_numpy(),
        "history": item_ids.map(history).fillna(0).to_numpy(),
        "rating": pd.to_numeric(item_frame["item_rating"], errors="coerce")
        .fillna(-1).to_numpy(),
        "reviews": pd.to_numeric(
            item_frame["item_rating_reviews_count"], errors="coerce"
        ).fillna(-1).to_numpy(),
        "location": normalize_identifier(item_frame["item_location_id"]),
    })
    ordered = frame.sort_values(
        ["history", "rating", "reviews", "item_id"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    global_order = ordered["item_index"].to_numpy(dtype=np.int32)
    assert len(global_order) == len(item_frame)
    assert np.unique(global_order).size == len(item_frame)
    by_location = {
        str(location): group["item_index"].to_numpy(dtype=np.int32)
        for location, group in ordered.groupby("location", sort=False)
    }
    return global_order, by_location


def fuse_local_global_top50(
    global_rankings: list[np.ndarray],
    location_rankings: list[np.ndarray],
    query_locations: np.ndarray,
    global_fallback: np.ndarray,
    location_fallback: dict[str, np.ndarray],
    local_quota: int,
    k: int = 50,
) -> list[np.ndarray]:
    '''Берёт квоту по локации, затем глобальную выдачу и резерв.'''
    final_rankings = []

    global_fallback = np.asarray(global_fallback, dtype=np.int32).ravel()

    for query_position, (global_ranking, local_ranking, query_location) in enumerate(zip(
        global_rankings,
        location_rankings,
        query_locations,
    )):
        selected = []
        seen = set()

        def add_until(indices, max_total):
            for raw_item_index in indices:
                if len(selected) >= max_total:
                    break

                item_index = int(raw_item_index)
                if item_index not in seen:
                    seen.add(item_index)
                    selected.append(item_index)

        add_until(local_ranking, max_total=min(local_quota, k))
        add_until(global_ranking, max_total=k)

        if len(selected) < k:
            add_until(location_fallback.get(str(query_location), ()), max_total=k)
        if len(selected) < k:
            add_until(global_fallback, max_total=k)

        if len(selected) != k:
            raise RuntimeError(
                f"Не удалось заполнить top-{k} для query_position={query_position}: "
                f"selected={len(selected)}, global_nonzero={len(global_ranking)}, "
                f"location_nonzero={len(local_ranking)}, "
                f"global_fallback={len(global_fallback)}, "
                f"location={query_location!r}"
            )

        assert len(set(selected)) == k
        final_rankings.append(np.asarray(selected, dtype=np.int32))

    return final_rankings


In [ ]:
b8_matrix = bm25f_matrices["title:description = 3:1"]
item_locations = normalize_identifier(validation_items["item_location_id"])
query_locations = normalize_identifier(validation_events["search_location_id"])
deep_retrieval = retrieve_global_and_location(
    query_vectors=field_queries,
    item_vectors=b8_matrix,
    query_locations=query_locations,
    item_locations=item_locations,
    item_tie_keys=item_tie_keys,
    global_k=1_000,
    location_k=500,
    batch_size=16,
)

depth_rows = []
for k in (50, 100, 200, 300, 500, 1_000):
    depth_rows.append({
        "k": k,
        "global_HitRate": calculate_hits(deep_retrieval["global_indices"], k).mean(),
        "location_HitRate": calculate_hits(deep_retrieval["location_indices"], k).mean(),
    })
display(pd.DataFrame(depth_rows).set_index("k").round(4))

global_fallback, location_fallback = build_fallback_orders(validation_items, fit_events)
quota_rows = []
for local_quota in (20, 30, 40, 50):
    rankings = fuse_local_global_top50(
        global_rankings=deep_retrieval["global_indices"],
        location_rankings=deep_retrieval["location_indices"],
        query_locations=query_locations,
        global_fallback=global_fallback,
        location_fallback=location_fallback,
        local_quota=local_quota,
        k=50,
    )
    quota_rows.append({
        "local_quota": local_quota,
        "HitRate@50": calculate_hits(rankings, 50).mean(),
    })
display(pd.DataFrame(quota_rows).set_index("local_quota").round(4))


Локация даёт основной прирост простого лексического решения. В финальной
цепочке сначала берутся 50 объявлений из локации: эта конфигурация служит основой для
последующего добавления кандидатов по микрокатегории и E5.


## 5. Не вошедшие в итоговую конфигурацию варианты

RRF с символьным TF-IDF не улучшил сильный BM25-бейзлайн. Поиск только
по `item_infm_params_text` оказался заметно слабее заголовка и описания,
а добавление параметров третьим полем давало небольшой нестабильный
прирост. Эти ветки не используются в итоговом решении.
